# CoralGuard-X — FINAL ACCURACY-FIRST

```text
RAW FRAME
    ↓
YOLO26m OLD DETECTOR
    ↓
Strong Candidate Boxes
    ↓
CoralSCOP
على كل candidate box
    ↓
Precise Instance Masks
    ↓
Overlap / Duplicate Resolver
    ↓
Masked Coral + 10% Context
    ↓
Dual-View EfficientNetV2-S
    ↓
Known Species / Unknown / Non-Coral
    ↓
SAM2.1 Temporal Memory
    ↓
Stable Final Video
```

**هدف النوتبوك:** تبدأ من الـcheckpoints والداتا الموجودة عندك محليًا، تجهز/تعيد استخدام CoralSCOP masks، تدرب Dual-View classifier، تعمل OOD calibration، ثم تختبر الـAccuracy-First pipeline على الفيديوهات وتحفظ فيديو H.264 نهائي.

- الـYOLO26m-SEG Student **مش مستخدم في الـmain path**؛ احتفظ به كـFast/Backup model.
- الـheld-out detector test split لا يُستخدم في التدريب أو threshold tuning.
- `Other/Unknown Coral` rejection عملي، وليس open-set benchmark رسمي.


In [ ]:
# CELL 1 — LOCAL ENVIRONMENT

from pathlib import Path
from collections import defaultdict
import os
import sys
import gc
import csv
import json
import math
import random
import shutil
import time
import zipfile
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from scipy.optimize import linear_sum_assignment

from ultralytics import YOLO
import ultralytics


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


WORKSPACE = Path(r"D:\Coral_Local_Workspace")
VENV_PYTHON = WORKSPACE / ".venv" / "Scripts" / "python.exe"

if not WORKSPACE.exists():
    raise FileNotFoundError(WORKSPACE)

if not VENV_PYTHON.exists():
    raise FileNotFoundError(VENV_PYTHON)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In VS Code select:\n"
        r"D:\Coral_Local_Workspace\.venv\Scripts\python.exe"
    )


DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)


# RTX A6000 48GB defaults.
if VRAM_GB >= 38:
    SEG_BATCH = 8
    CLS_BATCH = 64
elif VRAM_GB >= 22:
    SEG_BATCH = 6
    CLS_BATCH = 32
elif VRAM_GB >= 14:
    SEG_BATCH = 4
    CLS_BATCH = 16
else:
    SEG_BATCH = 2
    CLS_BATCH = 8


print("=" * 110)
print("CORALGUARD-X V2 LOCAL ENVIRONMENT")
print("=" * 110)
print("Python       :", sys.executable)
print("Workspace    :", WORKSPACE)
print("GPU          :", GPU_NAME)
print(f"VRAM         : {VRAM_GB:.2f} GB")
print("Torch        :", torch.__version__)
print("CUDA runtime :", torch.version.cuda)
print("Ultralytics  :", ultralytics.__version__)
print("timm         :", timm.__version__)
print("SEG batch    :", SEG_BATCH)
print("CLS batch    :", CLS_BATCH)
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# CELL 2 — ACCURACY-FIRST CONFIG

# Old detector candidate generation.
# Lower than the old final 0.375 operating point intentionally:
# CoralSCOP + classifier can reject some false candidates downstream.
DET_CANDIDATE_CONF = 0.20
DET_IMGSZ = 960
DET_NMS_IOU = 0.70
MAX_DET = 500

# CoralSCOP mask quality.
CORALSCOP_MIN_SCORE = 0.35
CORALSCOP_MIN_PIXELS = 40
MASK_BOX_EXPAND = 0.12

# Duplicate/overlap resolver.
DUP_MASK_IOU = 0.65
DUP_CONTAINMENT = 0.85
DUP_BOX_IOU_MIN = 0.35

# Dual-view context.
CONTEXT_PAD = 0.10

# Video refresh:
# 3 = accuracy-first practical mode.
# 1 = YOLO + CoralSCOP on every single frame (maximum compute).
REFRESH_STRIDE = 3
CLS_STRIDE = 3
SPECIES_EMA_ALPHA = 0.25

# Cross-refresh association.
TRACK_MAX_GAP = 12
TRACK_MASK_IOU_MIN = 0.12
TRACK_BOX_IOU_MIN = 0.10
MAX_SAM2_TRACKS = 60

SAM2_MODEL_ID = 'facebook/sam2.1-hiera-large'

# None = all videos / all frames.
VIDEO_LIMIT = None
VIDEO_MAX_FRAMES = None

print('DET_CANDIDATE_CONF =', DET_CANDIDATE_CONF)
print('REFRESH_STRIDE     =', REFRESH_STRIDE)
print('SAM2               =', SAM2_MODEL_ID)
print('STATUS: ACCURACY_CONFIG_READY')


In [ ]:
# CELL 3 — PATHS + VERIFIED OLD YOLO26m DETECTOR

PROJECT_ROOT = WORKSPACE / "data" / "Coral_Reef_AI_Project"

DET_ROOT = PROJECT_ROOT / "01_Data" / "Final" / "Detection_v1"

TRAIN_IMG = DET_ROOT / "images" / "train"
VAL_IMG = DET_ROOT / "images" / "validation"
TEST_IMG = DET_ROOT / "images" / "test"

TRAIN_COCO = DET_ROOT / "annotations" / "instances_train.json"
VAL_COCO = DET_ROOT / "annotations" / "instances_validation.json"

MANIFEST = (
    PROJECT_ROOT
    / "04_Dataset_Splits"
    / "classification_roi_manifest_split.csv"
)

MODELS_ROOT = WORKSPACE / "models"
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = WORKSPACE / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

FINAL_ROOT = WORKSPACE / "outputs" / "CoralGuard_X_ACCURACY_FIRST"

LEGACY_MASK_ROOT = (
    WORKSPACE
    / "outputs"
    / "CoralGuard_X_FINAL"
    / "01_CoralSCOP_Masks"
)

NEW_MASK_ROOT = FINAL_ROOT / "01_CoralSCOP_Teacher"

# Reuse masks already generated by the previous notebook if present.
if (LEGACY_MASK_ROOT / "coralscop_mask_manifest.csv").exists():
    MASK_ROOT = LEGACY_MASK_ROOT
    print("Reusing existing CoralSCOP masks:", MASK_ROOT)
else:
    MASK_ROOT = NEW_MASK_ROOT

SEG_DATA_ROOT = CACHE_ROOT / "Coral_YOLO26m_SEG_Student_Dataset"
SEG_ROOT = FINAL_ROOT / "_unused_seg_student"
DUAL_ROOT = FINAL_ROOT / "02_DualView_Data"
CLS_ROOT = FINAL_ROOT / "03_DualView_Classifier"
CAL_ROOT = FINAL_ROOT / "04_OOD_Calibration"
VIDEO_ROOT = FINAL_ROOT / "06_Final_Videos"

for d in [
    FINAL_ROOT,
    MASK_ROOT,
    SEG_ROOT,
    DUAL_ROOT,
    CLS_ROOT,
    CAL_ROOT,
    VIDEO_ROOT,
]:
    d.mkdir(parents=True, exist_ok=True)


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


# Optional manual override.
VERIFIED_OLD_DETECTOR = (
    WORKSPACE
    / "outputs"
    / "YOLO26m_EfficientNetV2S_FullPipeline"
    / "01_YOLO26m"
    / "Detection_YOLO26m"
    / "weights"
    / "best.pt"
)
MANUAL_DETECTOR_TEACHER = VERIFIED_OLD_DETECTOR if VERIFIED_OLD_DETECTOR.exists() else None
MANUAL_CLASSIFIER_TEACHER = None


DETECTOR_TEACHER_CANDIDATES = [
    MODELS_ROOT / "best_yolo26m_plus150.pt",
    MODELS_ROOT / "best_yolo26m_extended.pt",
    MODELS_ROOT / "best.pt",
    WORKSPACE
    / "outputs"
    / "YOLO26m_EfficientNetV2S_FullPipeline"
    / "01_YOLO26m"
    / "Detection_YOLO26m"
    / "weights"
    / "best.pt",
]

CLASSIFIER_TEACHER_CANDIDATES = [
    MODELS_ROOT / "best_efficientnetv2s_plus150.pth",
    MODELS_ROOT / "best_efficientnetv2s_macro_f1.pth",
    WORKSPACE
    / "outputs"
    / "YOLO26m_EfficientNetV2S_FullPipeline"
    / "02_EfficientNetV2S"
    / "best_efficientnetv2s_macro_f1.pth",
]


DETECTOR_TEACHER = (
    Path(MANUAL_DETECTOR_TEACHER)
    if MANUAL_DETECTOR_TEACHER
    else first_existing(DETECTOR_TEACHER_CANDIDATES)
)

CLASSIFIER_TEACHER = (
    Path(MANUAL_CLASSIFIER_TEACHER)
    if MANUAL_CLASSIFIER_TEACHER
    else first_existing(CLASSIFIER_TEACHER_CANDIDATES)
)


if DETECTOR_TEACHER is None:
    raise FileNotFoundError(
        "No YOLO26m detector checkpoint found. "
        "Put the strongest best.pt in D:\\Coral_Local_Workspace\\models"
    )

if CLASSIFIER_TEACHER is None:
    raise FileNotFoundError(
        "No EfficientNetV2-S checkpoint found. "
        "Put the strongest .pth in D:\\Coral_Local_Workspace\\models"
    )


required = [
    TRAIN_IMG,
    VAL_IMG,
    TRAIN_COCO,
    VAL_COCO,
    MANIFEST,
    DETECTOR_TEACHER,
    CLASSIFIER_TEACHER,
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(p)


print("=" * 110)
print("INPUTS")
print("=" * 110)
print("Detection teacher :", DETECTOR_TEACHER)
print("Classifier teacher:", CLASSIFIER_TEACHER)
print("Mask root         :", MASK_ROOT)
print("Final root        :", FINAL_ROOT)
print("Train             :", TRAIN_IMG)
print("Validation        :", VAL_IMG)
print("Test              : untouched")
print("STATUS: PATHS_READY")


In [ ]:
# CELL 4 — THIRD-PARTY SETUP: CORALSCOP + SAM2

THIRD_PARTY = WORKSPACE / "third_party"
THIRD_PARTY.mkdir(parents=True, exist_ok=True)

CORALSCOP_REPO = THIRD_PARTY / "CoralSCOP"
SAM2_REPO = THIRD_PARTY / "sam2"

GIT = shutil.which("git")

if GIT is None:
    raise RuntimeError(
        "Git for Windows is required and must be on PATH."
    )


if not CORALSCOP_REPO.exists():
    subprocess.check_call(
        [
            GIT,
            "clone",
            "https://github.com/zhengziqiang/CoralSCOP",
            str(CORALSCOP_REPO),
        ]
    )


if not SAM2_REPO.exists():
    subprocess.check_call(
        [
            GIT,
            "clone",
            "https://github.com/facebookresearch/sam2.git",
            str(SAM2_REPO),
        ]
    )


# Install SAM2 into the project venv only if import is unavailable.
try:
    import sam2
except Exception:
    print("Installing SAM2 into local .venv...")
    subprocess.check_call(
        [
            str(VENV_PYTHON),
            "-m",
            "pip",
            "install",
            "-e",
            str(SAM2_REPO),
        ]
    )


CORALSCOP_CKPT = MODELS_ROOT / "vit_b_coralscop.pth"

# Official checkpoint link used by the CoralSCOP project.
CORALSCOP_OFFICIAL_URL = (
    "https://www.dropbox.com/scl/fi/"
    "pw5jiq9oc8e8kvkx1fdk0/vit_b_coralscop.pth"
    "?rlkey=qczdohnzxwgwoadpzeht0lim2&dl=1"
)


if not CORALSCOP_CKPT.exists():
    import requests

    print("Downloading CoralSCOP ViT-B checkpoint...")

    with requests.get(
        CORALSCOP_OFFICIAL_URL,
        stream=True,
        timeout=120,
    ) as r:
        r.raise_for_status()

        with open(
            CORALSCOP_CKPT,
            "wb",
        ) as f:
            for chunk in r.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)


if (
    not CORALSCOP_CKPT.exists()
    or CORALSCOP_CKPT.stat().st_size
    < 50 * 1024 * 1024
):
    raise RuntimeError(
        "CoralSCOP checkpoint is missing/incomplete:\n"
        f"{CORALSCOP_CKPT}"
    )


print("CoralSCOP repo:", CORALSCOP_REPO)
print("SAM2 repo     :", SAM2_REPO)
print("CoralSCOP ckpt:", CORALSCOP_CKPT)
print("STATUS: THIRD_PARTY_READY")


In [ ]:
# CELL 5 — LOAD CORALSCOP

from pathlib import Path
import sys
import torch

repo_str = str(CORALSCOP_REPO)

if repo_str not in sys.path:
    sys.path.insert(
        0,
        repo_str,
    )

from segment_anything import (
    sam_model_registry,
    SamPredictor,
)


coralscop_model = sam_model_registry[
    "vit_b"
](
    checkpoint=str(
        CORALSCOP_CKPT
    )
)

coralscop_model.to(
    DEVICE
)

coralscop_model.eval()

coralscop_predictor = SamPredictor(
    coralscop_model
)


MASK_MIN_PIXELS = 40
MASK_MIN_SCORE = 0.35


print("CoralSCOP device:", next(coralscop_model.parameters()).device)
print("Predictor       :", type(coralscop_predictor).__name__)
print("MASK_MIN_SCORE  :", MASK_MIN_SCORE)
print("STATUS: CORALSCOP_TEACHER_READY")


In [ ]:
# CELL 6 — LOAD / GENERATE CORALSCOP GT-BOX MASKS

from collections import defaultdict
from pathlib import Path
import json

import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image


if "coralscop_predictor" not in globals():
    raise RuntimeError(
        "Run CELL 4 first: CoralSCOP teacher is not loaded."
    )


MASK_MANIFEST = MASK_ROOT / "coralscop_mask_manifest.csv"


def resolve_image(
    folder,
    file_name,
):
    p = folder / Path(
        file_name
    ).name

    if p.exists():
        return p

    matches = list(
        folder.glob(
            Path(
                file_name
            ).stem
            + ".*"
        )
    )

    if not matches:
        raise FileNotFoundError(
            file_name
        )

    return matches[
        0
    ]


def box_area(
    box,
):
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    return max(
        0.0,
        x2 - x1,
    ) * max(
        0.0,
        y2 - y1,
    )


def box_iou(
    a,
    b,
):
    ax1, ay1, ax2, ay2 = map(
        float,
        a,
    )

    bx1, by1, bx2, by2 = map(
        float,
        b,
    )

    ix1 = max(
        ax1,
        bx1,
    )

    iy1 = max(
        ay1,
        by1,
    )

    ix2 = min(
        ax2,
        bx2,
    )

    iy2 = min(
        ay2,
        by2,
    )

    inter = max(
        0.0,
        ix2 - ix1,
    ) * max(
        0.0,
        iy2 - iy1,
    )

    return inter / max(
        box_area(
            a
        )
        + box_area(
            b
        )
        - inter,
        1e-9,
    )


def mask_bbox(
    mask,
):
    ys, xs = np.where(
        mask
    )

    if len(
        xs
    ) == 0:
        return None

    return np.array(
        [
            xs.min(),
            ys.min(),
            xs.max() + 1,
            ys.max() + 1,
        ],
        dtype=np.float32,
    )


existing = {}

if MASK_MANIFEST.exists():
    old = pd.read_csv(
        MASK_MANIFEST
    )

    for _, r in old.iterrows():
        existing[
            (
                str(
                    r[
                        "split"
                    ]
                ),
                int(
                    r[
                        "ann_id"
                    ]
                ),
            )
        ] = dict(
            r
        )


rows = []


for (
    split,
    coco_path,
    img_folder,
) in [
    (
        "train",
        TRAIN_COCO,
        TRAIN_IMG,
    ),
    (
        "validation",
        VAL_COCO,
        VAL_IMG,
    ),
]:

    data = json.loads(
        coco_path.read_text(
            encoding="utf-8"
        )
    )

    by_img = defaultdict(
        list
    )

    for a in data[
        "annotations"
    ]:
        by_img[
            int(
                a[
                    "image_id"
                ]
            )
        ].append(
            a
        )

    info = {
        int(
            im[
                "id"
            ]
        ): im
        for im
        in data[
            "images"
        ]
    }

    out_dir = MASK_ROOT / split

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    for n, (
        image_id,
        im,
    ) in enumerate(
        info.items(),
        start=1,
    ):

        anns = by_img[
            image_id
        ]


        if all(
            (
                split,
                int(
                    a[
                        "id"
                    ]
                ),
            )
            in existing
            for a
            in anns
        ):

            rows.extend(
                existing[
                    (
                        split,
                        int(
                            a[
                                "id"
                            ]
                        ),
                    )
                ]
                for a
                in anns
            )

            continue


        path = resolve_image(
            img_folder,
            im[
                "file_name"
            ],
        )

        rgb = np.asarray(
            Image.open(
                path
            ).convert(
                "RGB"
            )
        )

        H, W = rgb.shape[
            :2
        ]

        # One image embedding, many box prompts.
        coralscop_predictor.set_image(
            rgb
        )


        for a in anns:

            key = (
                split,
                int(
                    a[
                        "id"
                    ]
                ),
            )


            if key in existing:
                rows.append(
                    existing[
                        key
                    ]
                )
                continue


            x, y, w, h = map(
                float,
                a[
                    "bbox"
                ],
            )

            gt_box = np.array(
                [
                    x,
                    y,
                    x + w,
                    y + h,
                ],
                dtype=np.float32,
            )

            gt_box[
                0::2
            ] = np.clip(
                gt_box[
                    0::2
                ],
                0,
                W - 1,
            )

            gt_box[
                1::2
            ] = np.clip(
                gt_box[
                    1::2
                ],
                0,
                H - 1,
            )


            with torch.inference_mode():

                masks, scores, _ = coralscop_predictor.predict(
                    box=gt_box,
                    multimask_output=True,
                )


            order = np.argsort(
                scores
            )[
                ::-1
            ]


            chosen = None
            chosen_score = -1.0


            for idx in order:

                m = masks[
                    idx
                ].astype(
                    bool
                )

                if (
                    m.sum()
                    >= MASK_MIN_PIXELS
                ):
                    chosen = m
                    chosen_score = float(
                        scores[
                            idx
                        ]
                    )

                    if (
                        chosen_score
                        >= MASK_MIN_SCORE
                    ):
                        break


            if chosen is None:
                raise RuntimeError(
                    f"CoralSCOP returned no usable mask: "
                    f"{split} ann={a['id']}"
                )


            mb = mask_bbox(
                chosen
            )

            if mb is None:
                raise RuntimeError(
                    f"Empty mask: {split} ann={a['id']}"
                )


            mask_area = float(
                chosen.sum()
            )

            gt_area = max(
                box_area(
                    gt_box
                ),
                1.0,
            )

            area_ratio = (
                mask_area
                / gt_area
            )

            bbox_iou = box_iou(
                mb,
                gt_box,
            )


            mask_path = (
                out_dir
                / f"ann_{int(a['id']):06d}.png"
            )

            cv2.imwrite(
                str(
                    mask_path
                ),
                chosen.astype(
                    np.uint8
                )
                * 255,
            )


            row = {
                "split": split,
                "ann_id": int(
                    a[
                        "id"
                    ]
                ),
                "image_id": image_id,
                "image_path": str(
                    path
                ),
                "image_stem": path.stem,
                "mask_path": str(
                    mask_path
                ),
                "x1": float(
                    gt_box[
                        0
                    ]
                ),
                "y1": float(
                    gt_box[
                        1
                    ]
                ),
                "x2": float(
                    gt_box[
                        2
                    ]
                ),
                "y2": float(
                    gt_box[
                        3
                    ]
                ),
                "mask_score": chosen_score,
                "mask_area_ratio_to_gt_box": area_ratio,
                "mask_bbox_iou_gt_box": bbox_iou,
            }


            rows.append(
                row
            )

            existing[
                key
            ] = row


        pd.DataFrame(
            rows
        ).drop_duplicates(
            [
                "split",
                "ann_id",
            ]
        ).to_csv(
            MASK_MANIFEST,
            index=False,
        )


        if (
            n
            % 25
            == 0
        ):
            print(
                f"{split}: {n}/{len(info)}"
            )


mask_df = pd.read_csv(
    MASK_MANIFEST
)


print()
print(
    mask_df.groupby(
        "split"
    ).size()
)

print(
    "Mean CoralSCOP score:",
    float(
        mask_df[
            "mask_score"
        ].mean()
    ),
)

print(
    "Manifest:",
    MASK_MANIFEST,
)

print("STATUS: TEACHER_MASKS_READY")


In [ ]:
# CELL 7 — BUILD DUAL-VIEW CLASSIFIER DATA

roi = pd.read_csv(
    MANIFEST
)

needed = {
    "roi_id",
    "image_stem",
    "raw_detection_label",
    "x1_px",
    "y1_px",
    "x2_px",
    "y2_px",
    "split",
}

missing = needed - set(
    roi.columns
)

if missing:
    raise RuntimeError(
        "Classification manifest schema mismatch: "
        + str(
            sorted(
                missing
            )
        )
    )


mask_index = defaultdict(
    list
)

for _, r in mask_df.iterrows():
    mask_index[
        (
            str(
                r[
                    "split"
                ]
            ),
            str(
                r[
                    "image_stem"
                ]
            ),
        )
    ].append(
        r
    )


positive_rows = []


for _, r in roi[
    roi[
        "split"
    ].astype(
        str
    ).isin(
        [
            "train",
            "validation",
        ]
    )
].iterrows():

    key = (
        str(
            r[
                "split"
            ]
        ),
        str(
            r[
                "image_stem"
            ]
        ),
    )

    candidates = mask_index[
        key
    ]

    q = [
        float(
            r[
                "x1_px"
            ]
        ),
        float(
            r[
                "y1_px"
            ]
        ),
        float(
            r[
                "x2_px"
            ]
        ),
        float(
            r[
                "y2_px"
            ]
        ),
    ]

    best = max(
        candidates,
        key=lambda z: box_iou(
            q,
            [
                z[
                    "x1"
                ],
                z[
                    "y1"
                ],
                z[
                    "x2"
                ],
                z[
                    "y2"
                ],
            ],
        ),
    )

    mapped_iou = box_iou(
        q,
        [
            best[
                "x1"
            ],
            best[
                "y1"
            ],
            best[
                "x2"
            ],
            best[
                "y2"
            ],
        ],
    )

    if mapped_iou < 0.90:
        raise RuntimeError(
            f"Mask mapping failed for {r['roi_id']}: {mapped_iou:.3f}"
        )

    positive_rows.append(
        {
            "roi_id": str(
                r[
                    "roi_id"
                ]
            ),
            "split": str(
                r[
                    "split"
                ]
            ),
            "label": str(
                r[
                    "raw_detection_label"
                ]
            ),
            "image_path": str(
                best[
                    "image_path"
                ]
            ),
            "mask_path": str(
                best[
                    "mask_path"
                ]
            ),
            "x1": q[
                0
            ],
            "y1": q[
                1
            ],
            "x2": q[
                2
            ],
            "y2": q[
                3
            ],
        }
    )


pos = pd.DataFrame(
    positive_rows
)


assert (
    pos[
        "split"
    ]
    == "train"
).sum() == 1540

assert (
    pos[
        "split"
    ]
    == "validation"
).sum() == 354


CONTEXT_PAD = 0.10

PAIR_MANIFEST = (
    DUAL_ROOT
    / "dual_view_manifest.csv"
)


def padded_box(
    box,
    W,
    H,
    pad=0.10,
):
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    bw = max(
        1.0,
        x2 - x1,
    )

    bh = max(
        1.0,
        y2 - y1,
    )

    px = bw * pad
    py = bh * pad

    return (
        max(
            0,
            int(
                math.floor(
                    x1 - px
                )
            ),
        ),
        max(
            0,
            int(
                math.floor(
                    y1 - py
                )
            ),
        ),
        min(
            W,
            int(
                math.ceil(
                    x2 + px
                )
            ),
        ),
        min(
            H,
            int(
                math.ceil(
                    y2 + py
                )
            ),
        ),
    )


pair_rows = []


for _, r in pos.iterrows():

    out_dir = (
        DUAL_ROOT
        / r[
            "split"
        ]
        / r[
            "label"
        ]
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cpath = (
        out_dir
        / f"{r['roi_id']}__context.jpg"
    )

    mpath = (
        out_dir
        / f"{r['roi_id']}__masked.jpg"
    )


    if not (
        cpath.exists()
        and mpath.exists()
    ):

        rgb = np.asarray(
            Image.open(
                r[
                    "image_path"
                ]
            ).convert(
                "RGB"
            )
        )

        mask = (
            cv2.imread(
                r[
                    "mask_path"
                ],
                cv2.IMREAD_GRAYSCALE,
            )
            > 127
        )

        H, W = rgb.shape[
            :2
        ]

        x1, y1, x2, y2 = padded_box(
            [
                r[
                    "x1"
                ],
                r[
                    "y1"
                ],
                r[
                    "x2"
                ],
                r[
                    "y2"
                ],
            ],
            W,
            H,
            CONTEXT_PAD,
        )

        context = rgb[
            y1:y2,
            x1:x2,
        ].copy()

        local_mask = mask[
            y1:y2,
            x1:x2,
        ]

        masked = context.copy()

        masked[
            ~local_mask
        ] = 0


        Image.fromarray(
            context
        ).save(
            cpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )

        Image.fromarray(
            masked
        ).save(
            mpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )


    pair_rows.append(
        {
            "split": r[
                "split"
            ],
            "label": r[
                "label"
            ],
            "context_path": str(
                cpath
            ),
            "masked_path": str(
                mpath
            ),
            "source": "CORAL",
        }
    )


# ------------------------------------------------------------
# Real NON_CORAL background negatives outside GT boxes.
# ------------------------------------------------------------

def coco_boxes(
    coco_path,
):
    data = json.loads(
        coco_path.read_text(
            encoding="utf-8"
        )
    )

    info = {
        int(
            im[
                "id"
            ]
        ): im
        for im
        in data[
            "images"
        ]
    }

    boxes = defaultdict(
        list
    )

    for a in data[
        "annotations"
    ]:
        x, y, w, h = map(
            float,
            a[
                "bbox"
            ],
        )

        boxes[
            int(
                a[
                    "image_id"
                ]
            )
        ].append(
            [
                x,
                y,
                x + w,
                y + h,
            ]
        )

    return (
        info,
        boxes,
    )


rng = np.random.default_rng(
    SEED
)


for (
    split,
    coco_path,
    img_folder,
) in [
    (
        "train",
        TRAIN_COCO,
        TRAIN_IMG,
    ),
    (
        "validation",
        VAL_COCO,
        VAL_IMG,
    ),
]:

    info, boxes_by_img = coco_boxes(
        coco_path
    )

    out_dir = (
        DUAL_ROOT
        / split
        / "NON_CORAL"
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    for image_id, im in info.items():

        path = resolve_image(
            img_folder,
            im[
                "file_name"
            ],
        )

        rgb = np.asarray(
            Image.open(
                path
            ).convert(
                "RGB"
            )
        )

        H, W = rgb.shape[
            :2
        ]

        q = None


        for _ in range(
            150
        ):

            side = int(
                max(
                    48,
                    min(
                        W,
                        H,
                    )
                    * float(
                        rng.uniform(
                            0.10,
                            0.32,
                        )
                    ),
                )
            )

            if (
                W <= side
                or H <= side
            ):
                continue

            x1 = int(
                rng.integers(
                    0,
                    W - side,
                )
            )

            y1 = int(
                rng.integers(
                    0,
                    H - side,
                )
            )

            cand = [
                x1,
                y1,
                x1 + side,
                y1 + side,
            ]

            mx = max(
                [
                    box_iou(
                        cand,
                        b,
                    )
                    for b
                    in boxes_by_img[
                        image_id
                    ]
                ]
                + [
                    0.0
                ]
            )

            if mx < 0.02:
                q = cand
                break


        if q is None:
            continue


        x1, y1, x2, y2 = q

        context = rgb[
            y1:y2,
            x1:x2,
        ].copy()

        masked = np.zeros_like(
            context
        )

        stem = Path(
            im[
                "file_name"
            ]
        ).stem

        cpath = (
            out_dir
            / f"{stem}__neg__context.jpg"
        )

        mpath = (
            out_dir
            / f"{stem}__neg__masked.jpg"
        )


        Image.fromarray(
            context
        ).save(
            cpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )

        Image.fromarray(
            masked
        ).save(
            mpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )


        pair_rows.append(
            {
                "split": split,
                "label": "NON_CORAL",
                "context_path": str(
                    cpath
                ),
                "masked_path": str(
                    mpath
                ),
                "source": "BACKGROUND_NEGATIVE",
            }
        )


pair_df = pd.DataFrame(
    pair_rows
)

pair_df.to_csv(
    PAIR_MANIFEST,
    index=False,
)


print(
    pair_df.groupby(
        [
            "split",
            "label",
        ]
    ).size()
)

print("Manifest:", PAIR_MANIFEST)
print("STATUS: DUAL_VIEW_DATA_READY")


In [ ]:
# CELL 8 — DUAL-VIEW EFFICIENTNETV2-S + TRANSFER

old_ckpt = torch.load(
    CLASSIFIER_TEACHER,
    map_location=DEVICE,
    weights_only=False,
)


MODEL_NAME = old_ckpt.get(
    "model_name",
    "tf_efficientnetv2_s.in21k_ft_in1k",
)

OLD_CLASS_TO_IDX = old_ckpt[
    "class_to_idx"
]

KNOWN_CLASSES = [
    x
    for x, _
    in sorted(
        OLD_CLASS_TO_IDX.items(),
        key=lambda kv: kv[
            1
        ],
    )
]

ALL_CLASSES = (
    KNOWN_CLASSES
    + [
        "NON_CORAL"
    ]
)

CLASS_TO_IDX = {
    c: i
    for i, c
    in enumerate(
        ALL_CLASSES
    )
}

IDX_TO_CLASS = {
    i: c
    for c, i
    in CLASS_TO_IDX.items()
}

NCLS = len(
    ALL_CLASSES
)


CLS_SIZE = int(
    old_ckpt.get(
        "input_size",
        300,
    )
)

MEAN = tuple(
    old_ckpt.get(
        "mean",
        (
            0.5,
            0.5,
            0.5,
        ),
    )
)

STD = tuple(
    old_ckpt.get(
        "std",
        (
            0.5,
            0.5,
            0.5,
        ),
    )
)


train_tf = transforms.Compose(
    [
        transforms.Resize(
            (
                CLS_SIZE,
                CLS_SIZE,
            ),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.RandomHorizontalFlip(
            0.5
        ),
        transforms.RandomVerticalFlip(
            0.08
        ),
        transforms.RandomRotation(
            4,
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.06,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            MEAN,
            STD,
        ),
    ]
)


val_tf = transforms.Compose(
    [
        transforms.Resize(
            (
                CLS_SIZE,
                CLS_SIZE,
            ),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            MEAN,
            STD,
        ),
    ]
)


class DualPairDataset(
    Dataset
):
    def __init__(
        self,
        df,
        transform,
    ):
        self.df = df.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(
        self
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        idx,
    ):
        r = self.df.iloc[
            idx
        ]

        context = Image.open(
            r[
                "context_path"
            ]
        ).convert(
            "RGB"
        )

        masked = Image.open(
            r[
                "masked_path"
            ]
        ).convert(
            "RGB"
        )

        context = self.transform(
            context
        )

        masked = self.transform(
            masked
        )

        y = CLASS_TO_IDX[
            str(
                r[
                    "label"
                ]
            )
        ]

        return (
            context,
            masked,
            y,
        )


train_pairs = pair_df[
    pair_df[
        "split"
    ]
    == "train"
].copy()

val_pairs = pair_df[
    pair_df[
        "split"
    ]
    == "validation"
].copy()


train_ds = DualPairDataset(
    train_pairs,
    train_tf,
)

train_eval_ds = DualPairDataset(
    train_pairs,
    val_tf,
)

val_ds = DualPairDataset(
    val_pairs,
    val_tf,
)


labels_train = np.asarray(
    [
        CLASS_TO_IDX[
            str(
                x
            )
        ]
        for x
        in train_pairs[
            "label"
        ].tolist()
    ],
    dtype=np.int64,
)


counts = np.bincount(
    labels_train,
    minlength=NCLS,
).astype(
    np.float32
)

class_weights_np = np.sqrt(
    counts.max()
    / np.maximum(
        counts,
        1.0,
    )
)

class_weights_np /= class_weights_np.mean()


sample_weights = class_weights_np[
    labels_train
]


sampler = WeightedRandomSampler(
    weights=torch.tensor(
        sample_weights,
        dtype=torch.double,
    ),
    num_samples=len(
        sample_weights
    ),
    replacement=True,
)


train_loader = DataLoader(
    train_ds,
    batch_size=CLS_BATCH,
    sampler=sampler,
    num_workers=0,
    pin_memory=True,
)

train_eval_loader = DataLoader(
    train_eval_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


class DualViewEfficientNet(
    nn.Module
):
    def __init__(
        self,
        model_name,
        num_classes,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        feat_dim = int(
            self.backbone.num_features
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                feat_dim
                * 2,
                feat_dim,
            ),
            nn.GELU(),
            nn.LayerNorm(
                feat_dim
            ),
            nn.Dropout(
                0.20
            ),
        )

        self.head = nn.Linear(
            feat_dim,
            num_classes,
        )

        self.feat_dim = feat_dim

    def forward(
        self,
        context,
        masked,
        return_embedding=False,
    ):
        f_context = self.backbone(
            context
        )

        f_masked = self.backbone(
            masked
        )

        fused = torch.cat(
            [
                f_context,
                f_masked,
            ],
            dim=1,
        )

        emb = self.fusion(
            fused
        )

        logits = self.head(
            emb
        )

        if return_embedding:
            return (
                logits,
                emb,
            )

        return logits


dual_model = DualViewEfficientNet(
    MODEL_NAME,
    NCLS,
).to(
    DEVICE
)


# Transfer all matching backbone tensors from the existing classifier.
old_state = old_ckpt[
    "model_state_dict"
]

new_backbone_state = (
    dual_model.backbone.state_dict()
)

transfer = {}


for k, v in old_state.items():

    if (
        k
        in new_backbone_state
        and new_backbone_state[
            k
        ].shape
        == v.shape
    ):
        transfer[
            k
        ] = v


dual_model.backbone.load_state_dict(
    transfer,
    strict=False,
)


print(
    "Transferred backbone tensors:",
    len(
        transfer
    ),
    "/",
    len(
        new_backbone_state
    ),
)

print("Classes:", CLASS_TO_IDX)
print("STATUS: DUAL_VIEW_MODEL_READY")


In [ ]:
# CELL 9 — TRAIN DUAL-VIEW CLASSIFIER (UP TO 150 EPOCHS)

DUAL_EPOCHS = 150
DUAL_LR = 1e-5
PATIENCE = 30


BEST_DUAL = (
    CLS_ROOT
    / "best_dualview_effnetv2s_macro_f1.pth"
)

LAST_DUAL = (
    CLS_ROOT
    / "last_dualview_effnetv2s.pth"
)

HISTORY_CSV = (
    CLS_ROOT
    / "dualview_training_history.csv"
)


criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(
        class_weights_np,
        dtype=torch.float32,
        device=DEVICE,
    ),
    label_smoothing=0.03,
)


optimizer = torch.optim.AdamW(
    dual_model.parameters(),
    lr=DUAL_LR,
    weight_decay=1e-4,
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=DUAL_EPOCHS,
    eta_min=2e-7,
)


scaler = torch.amp.GradScaler(
    "cuda"
)


def evaluate_dual(
    model,
    loader,
    return_embeddings=False,
):
    model.eval()

    yt = []
    yp = []
    probs_all = []
    emb_all = []

    with torch.no_grad():

        for context, masked, y in loader:

            context = context.to(
                DEVICE,
                non_blocking=True,
            )

            masked = masked.to(
                DEVICE,
                non_blocking=True,
            )

            y = torch.as_tensor(
                y,
                device=DEVICE,
            )


            with torch.autocast(
                "cuda",
                dtype=torch.float16,
            ):

                logits, emb = model(
                    context,
                    masked,
                    return_embedding=True,
                )


            probs = torch.softmax(
                logits.float(),
                dim=1,
            )

            pred = probs.argmax(
                dim=1
            )


            yt.extend(
                y.cpu().tolist()
            )

            yp.extend(
                pred.cpu().tolist()
            )

            probs_all.append(
                probs.cpu().numpy()
            )


            if return_embeddings:
                emb_all.append(
                    emb.float().cpu().numpy()
                )


    yt = np.asarray(
        yt
    )

    yp = np.asarray(
        yp
    )

    probs_all = np.concatenate(
        probs_all,
        axis=0,
    )


    metrics = {
        "accuracy": accuracy_score(
            yt,
            yp,
        ),
        "macro_precision": precision_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            yt,
            yp,
            average="weighted",
            zero_division=0,
        ),
    }


    if return_embeddings:
        emb_all = np.concatenate(
            emb_all,
            axis=0,
        )

        return (
            metrics,
            yt,
            yp,
            probs_all,
            emb_all,
        )


    return (
        metrics,
        yt,
        yp,
        probs_all,
    )


history = []

best_f1 = -1.0
best_epoch = -1
no_improve = 0


for epoch in range(
    1,
    DUAL_EPOCHS + 1,
):

    dual_model.train()

    total_loss = 0.0
    seen = 0


    for context, masked, y in train_loader:

        context = context.to(
            DEVICE,
            non_blocking=True,
        )

        masked = masked.to(
            DEVICE,
            non_blocking=True,
        )

        y = torch.as_tensor(
            y,
            device=DEVICE,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            "cuda",
            dtype=torch.float16,
        ):

            logits = dual_model(
                context,
                masked,
            )

            loss = criterion(
                logits,
                y,
            )


        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            dual_model.parameters(),
            5.0,
        )

        scaler.step(
            optimizer
        )

        scaler.update()


        total_loss += (
            float(
                loss.detach().cpu()
            )
            * context.size(
                0
            )
        )

        seen += context.size(
            0
        )


    scheduler.step()


    val_metrics, _, _, _ = evaluate_dual(
        dual_model,
        val_loader,
    )


    row = {
        "epoch": epoch,
        "train_loss": total_loss
        / max(
            seen,
            1,
        ),
        "lr": optimizer.param_groups[
            0
        ][
            "lr"
        ],
        **{
            f"val_{k}": float(
                v
            )
            for k, v
            in val_metrics.items()
        },
    }


    history.append(
        row
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_CSV,
        index=False,
    )


    payload = {
        "model_state_dict": dual_model.state_dict(),
        "model_name": MODEL_NAME,
        "class_to_idx": CLASS_TO_IDX,
        "known_classes": KNOWN_CLASSES,
        "input_size": CLS_SIZE,
        "context_padding": CONTEXT_PAD,
        "mean": MEAN,
        "std": STD,
        "epoch": epoch,
        "val_metrics": {
            k: float(
                v
            )
            for k, v
            in val_metrics.items()
        },
        "architecture": "DualViewEfficientNetV2S",
    }


    torch.save(
        payload,
        LAST_DUAL,
    )


    if (
        val_metrics[
            "macro_f1"
        ]
        > best_f1
    ):

        best_f1 = float(
            val_metrics[
                "macro_f1"
            ]
        )

        best_epoch = epoch

        no_improve = 0

        torch.save(
            payload,
            BEST_DUAL,
        )

    else:
        no_improve += 1


    print(
        f"Epoch {epoch:03d}/{DUAL_EPOCHS}"
        f" | loss={row['train_loss']:.4f}"
        f" | Acc={val_metrics['accuracy']:.4f}"
        f" | MacroF1={val_metrics['macro_f1']:.4f}"
        f" | Best={best_f1:.4f}"
    )


    if no_improve >= PATIENCE:
        print(
            f"Early stopping after {PATIENCE} epochs without improvement."
        )
        break


print("Best epoch   :", best_epoch)
print("Best Macro-F1:", best_f1)
print("Best model   :", BEST_DUAL)
print("STATUS: DUAL_VIEW_TRAINED")


In [ ]:
# CELL 10 — OOD CALIBRATION

best_ckpt = torch.load(
    BEST_DUAL,
    map_location=DEVICE,
    weights_only=False,
)

dual_model.load_state_dict(
    best_ckpt[
        "model_state_dict"
    ]
)

dual_model.eval()


(
    train_metrics,
    train_true,
    train_pred,
    train_probs,
    train_emb,
) = evaluate_dual(
    dual_model,
    train_eval_loader,
    return_embeddings=True,
)


(
    val_metrics,
    val_true,
    val_pred,
    val_probs,
    val_emb,
) = evaluate_dual(
    dual_model,
    val_loader,
    return_embeddings=True,
)


train_emb = train_emb / np.maximum(
    np.linalg.norm(
        train_emb,
        axis=1,
        keepdims=True,
    ),
    1e-9,
)

val_emb = val_emb / np.maximum(
    np.linalg.norm(
        val_emb,
        axis=1,
        keepdims=True,
    ),
    1e-9,
)


PROTOTYPES = {}
SIM_THRESHOLDS = {}


for class_name in KNOWN_CLASSES:

    ci = CLASS_TO_IDX[
        class_name
    ]

    idx_train = np.where(
        train_true
        == ci
    )[
        0
    ]

    if len(
        idx_train
    ) == 0:
        continue


    proto = train_emb[
        idx_train
    ].mean(
        axis=0
    )

    proto /= max(
        np.linalg.norm(
            proto
        ),
        1e-9,
    )

    PROTOTYPES[
        class_name
    ] = proto.astype(
        np.float32
    )


    idx_val = np.where(
        val_true
        == ci
    )[
        0
    ]

    if len(
        idx_val
    ) > 0:

        sims = val_emb[
            idx_val
        ] @ proto

        SIM_THRESHOLDS[
            class_name
        ] = float(
            np.percentile(
                sims,
                3.0,
            )
        )


PROTOTYPE_NPZ = (
    CAL_ROOT
    / "ood_prototypes.npz"
)

np.savez(
    PROTOTYPE_NPZ,
    **{
        f"proto__{k}": v
        for k, v
        in PROTOTYPES.items()
    },
)


CALIBRATION_JSON = (
    CAL_ROOT
    / "ood_calibration.json"
)


calibration = {
    "known_classes": KNOWN_CLASSES,
    "class_to_idx": CLASS_TO_IDX,
    "similarity_thresholds": SIM_THRESHOLDS,
    "known_conf_threshold": 0.50,
    "known_margin_threshold": 0.08,
    "non_coral_threshold": 0.45,
    "note": (
        "Other/Unknown Coral is a pragmatic open-set rejection. "
        "There is no explicit unseen-species validation set."
    ),
}


CALIBRATION_JSON.write_text(
    json.dumps(
        calibration,
        indent=2,
    ),
    encoding="utf-8",
)


report = pd.DataFrame(
    classification_report(
        val_true,
        val_pred,
        target_names=[
            IDX_TO_CLASS[
                i
            ]
            for i
            in range(
                NCLS
            )
        ],
        zero_division=0,
        output_dict=True,
    )
).T


report.to_csv(
    CLS_ROOT
    / "final_validation_classification_report.csv"
)


np.savetxt(
    CLS_ROOT
    / "final_validation_confusion_matrix.csv",
    confusion_matrix(
        val_true,
        val_pred,
    ),
    fmt="%d",
    delimiter=",",
)


print("=" * 110)
print("FINAL CLASSIFIER VALIDATION")
print("=" * 110)

for k, v in val_metrics.items():
    print(
        f"{k:18s}: {v:.6f}"
    )

print("Calibration:", CALIBRATION_JSON)
print("STATUS: OOD_READY")


In [ ]:
# CELL 11 — FINAL ACCURACY-FIRST INFERENCE STACK

# Reload strongest old detector as the primary detector.
detector = YOLO(str(DETECTOR_TEACHER))

best_ckpt = torch.load(
    BEST_DUAL,
    map_location=DEVICE,
    weights_only=False,
)
dual_model.load_state_dict(best_ckpt['model_state_dict'])
dual_model.eval()

cal = json.loads(
    CALIBRATION_JSON.read_text(encoding='utf-8')
)
proto_file = np.load(PROTOTYPE_NPZ)
PROTOTYPES = {
    k.replace('proto__', ''): proto_file[k]
    for k in proto_file.files
}


def normalize_mask_2d(mask):
    if mask is None:
        raise ValueError('Mask is None')
    mask = np.asarray(mask)
    mask = np.squeeze(mask)
    if mask.ndim == 3:
        if mask.shape[0] in (1, 3, 4):
            mask = mask[0]
        elif mask.shape[-1] in (1, 3, 4):
            mask = mask[..., 0]
    if mask.ndim != 2:
        raise ValueError(f'Expected 2D mask, got {mask.shape}')
    return (mask > 0).astype(bool)


def safe_mask_bbox(mask):
    mask = normalize_mask_2d(mask)
    ys, xs = np.where(mask)
    if xs.size == 0:
        return None
    return np.array(
        [xs.min(), ys.min(), xs.max()+1, ys.max()+1],
        dtype=np.float32,
    )


def mask_iou(a, b):
    a = normalize_mask_2d(a)
    b = normalize_mask_2d(b)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / max(union, 1))


def mask_containment(a, b):
    a = normalize_mask_2d(a)
    b = normalize_mask_2d(b)
    inter = np.logical_and(a, b).sum()
    smaller = min(int(a.sum()), int(b.sum()))
    return float(inter / max(smaller, 1))


def expand_box(box, W, H, pad=0.10):
    x1, y1, x2, y2 = map(float, box)
    bw = max(1.0, x2-x1)
    bh = max(1.0, y2-y1)
    px, py = bw*pad, bh*pad
    return np.array([
        max(0.0, x1-px),
        max(0.0, y1-py),
        min(float(W-1), x2+px),
        min(float(H-1), y2+py),
    ], dtype=np.float32)


def padded_int_box(box, W, H, pad=0.10):
    b = expand_box(box, W, H, pad)
    return (
        max(0, int(math.floor(b[0]))),
        max(0, int(math.floor(b[1]))),
        min(W, int(math.ceil(b[2]))),
        min(H, int(math.ceil(b[3]))),
    )


def clip_mask_to_box(mask, box, pad=0.12):
    mask = normalize_mask_2d(mask)
    H, W = mask.shape
    x1, y1, x2, y2 = padded_int_box(box, W, H, pad)
    out = np.zeros_like(mask, dtype=bool)
    out[y1:y2, x1:x2] = mask[y1:y2, x1:x2]
    return out


def choose_coralscop_mask(masks, scores, det_box):
    best = None
    for i in range(len(scores)):
        m = normalize_mask_2d(masks[i])
        m = clip_mask_to_box(m, det_box, MASK_BOX_EXPAND)
        if int(m.sum()) < CORALSCOP_MIN_PIXELS:
            continue

        mb = safe_mask_bbox(m)
        if mb is None:
            continue

        sam_score = float(scores[i])
        align = float(box_iou(mb, det_box))
        area_ratio = float(m.sum()) / max(box_area(det_box), 1.0)
        ratio_term = math.exp(-abs(math.log(max(area_ratio, 1e-6))))

        quality = (
            0.55 * sam_score
            + 0.30 * align
            + 0.15 * ratio_term
        )

        cand = {
            'mask': m,
            'mask_score': sam_score,
            'mask_quality': float(quality),
            'mask_bbox': mb,
            'mask_box_iou': align,
            'mask_area_ratio': area_ratio,
        }
        if best is None or cand['mask_quality'] > best['mask_quality']:
            best = cand
    return best


def resolve_mask_duplicates(instances):
    ordered = sorted(
        instances,
        key=lambda x: 0.55*x['det_conf'] + 0.45*x['mask_quality'],
        reverse=True,
    )
    kept = []
    for inst in ordered:
        dup = False
        for k in kept:
            mi = mask_iou(inst['mask'], k['mask'])
            contain = mask_containment(inst['mask'], k['mask'])
            bi = box_iou(inst['det_box'], k['det_box'])
            if mi >= DUP_MASK_IOU or (
                contain >= DUP_CONTAINMENT
                and bi >= DUP_BOX_IOU_MIN
            ):
                dup = True
                break
        if not dup:
            kept.append(inst)
    return kept


def detect_and_coralscop(rgb):
    H, W = rgb.shape[:2]
    r = detector.predict(
        source=rgb,
        imgsz=DET_IMGSZ,
        conf=DET_CANDIDATE_CONF,
        iou=DET_NMS_IOU,
        max_det=MAX_DET,
        device=0,
        verbose=False,
    )[0]

    if r.boxes is None or len(r.boxes) == 0:
        return []

    boxes = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    confs = r.boxes.conf.detach().cpu().numpy().astype(np.float32)

    # One expensive image embedding, then all YOLO boxes are prompts.
    coralscop_predictor.set_image(rgb)

    out = []
    for box, det_conf in zip(boxes, confs):
        box = box.astype(np.float32)
        box[0::2] = np.clip(box[0::2], 0, W-1)
        box[1::2] = np.clip(box[1::2], 0, H-1)

        with torch.inference_mode():
            masks, scores, _ = coralscop_predictor.predict(
                box=box,
                multimask_output=True,
            )

        chosen = choose_coralscop_mask(masks, scores, box)
        if chosen is None:
            continue

        out.append({
            'det_box': box,
            'box': chosen['mask_bbox'],
            'mask': chosen['mask'],
            'det_conf': float(det_conf),
            'mask_score': chosen['mask_score'],
            'mask_quality': chosen['mask_quality'],
            'mask_box_iou': chosen['mask_box_iou'],
            'mask_area_ratio': chosen['mask_area_ratio'],
        })

    return resolve_mask_duplicates(out)


def make_dual_views(rgb, mask, box):
    H, W = rgb.shape[:2]
    x1, y1, x2, y2 = padded_int_box(box, W, H, CONTEXT_PAD)
    if x2 <= x1 or y2 <= y1:
        return None, None

    context = rgb[y1:y2, x1:x2].copy()
    local_mask = mask[y1:y2, x1:x2]
    if context.size == 0 or int(local_mask.sum()) == 0:
        return None, None

    masked = context.copy()
    masked[~local_mask] = 0
    return Image.fromarray(context), Image.fromarray(masked)


def classify_instances(rgb, instances):
    if not instances:
        return []

    valid, contexts, maskeds = [], [], []
    for inst in instances:
        context, masked = make_dual_views(rgb, inst['mask'], inst['box'])
        if context is None:
            continue
        valid.append(inst)
        contexts.append(val_tf(context))
        maskeds.append(val_tf(masked))

    if not valid:
        return []

    ct = torch.stack(contexts).to(DEVICE)
    mt = torch.stack(maskeds).to(DEVICE)

    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        logits, emb = dual_model(ct, mt, return_embedding=True)

    probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
    emb = emb.float().cpu().numpy()
    emb /= np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-9)

    for inst, p, e in zip(valid, probs, emb):
        order = np.argsort(p)[::-1]
        top, second = int(order[0]), int(order[1])
        raw = IDX_TO_CLASS[top]
        conf = float(p[top])
        margin = float(p[top] - p[second])
        non_prob = float(p[CLASS_TO_IDX['NON_CORAL']])

        if raw == 'NON_CORAL' or non_prob >= float(cal['non_coral_threshold']):
            label = 'Non-Coral'
            similarity = None
        else:
            proto = PROTOTYPES.get(raw)
            similarity = float(e @ proto) if proto is not None else -1.0
            if (
                conf < float(cal['known_conf_threshold'])
                or margin < float(cal['known_margin_threshold'])
                or similarity < float(cal['similarity_thresholds'].get(raw, -1.0))
            ):
                label = 'Other/Unknown Coral'
            else:
                label = raw

        inst['label'] = label
        inst['cls_conf'] = conf
        inst['cls_margin'] = margin
        inst['prob_vector'] = p.astype(np.float32)
        inst['similarity'] = similarity

    return valid


def final_spatial_instances(rgb):
    return classify_instances(rgb, detect_and_coralscop(rgb))

print('Detector   :', DETECTOR_TEACHER)
print('CoralSCOP  :', CORALSCOP_CKPT)
print('Dual-View  :', BEST_DUAL)
print('STATUS: ACCURACY_FIRST_STACK_READY')


In [ ]:
# CELL 12 — ONE-IMAGE SMOKE TEST

SMOKE_ROOT = FINAL_ROOT / '05_Smoke_Test'
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

val_images = sorted([
    p for p in VAL_IMG.iterdir()
    if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}
])
if not val_images:
    raise RuntimeError('No validation images found')

smoke_path = val_images[0]
bgr = cv2.imread(str(smoke_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

t0 = time.time()
instances = final_spatial_instances(rgb)
elapsed = time.time() - t0

vis = bgr.copy()
overlay = bgr.copy()
for i, inst in enumerate(instances, 1):
    label = inst['label']
    color = (150,150,150) if label == 'Non-Coral' else ((0,165,255) if label == 'Other/Unknown Coral' else (0,255,0))
    mask = inst['mask']
    overlay[mask] = (0.55*overlay[mask] + 0.45*np.asarray(color,dtype=np.float32)).astype(np.uint8)
    contours,_ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, contours, -1, color, 2)
    x1,y1,x2,y2 = [int(round(z)) for z in inst['box']]
    txt = f"{i}: {label} | D={inst['det_conf']:.2f} C={inst['cls_conf']:.2f} M={inst['mask_score']:.2f}"
    cv2.putText(vis, txt, (x1,max(25,y1-7)), cv2.FONT_HERSHEY_SIMPLEX, 0.58, color, 2, cv2.LINE_AA)

vis = cv2.addWeighted(overlay,0.42,vis,0.58,0)
out = SMOKE_ROOT / f'{smoke_path.stem}_accuracy_first.jpg'
cv2.imwrite(str(out), vis, [int(cv2.IMWRITE_JPEG_QUALITY),95])

print('Input:', smoke_path)
print('Instances:', len(instances))
print(f'Time: {elapsed:.2f}s')
print('Saved:', out)
print('STATUS: SMOKE_TEST_COMPLETE')


In [ ]:
# CELL 13 — VIDEO INPUT + ASSOCIATION HELPERS

VIDEO_ZIP = WORKSPACE / 'data' / 'Coral_video' / 'coral_video.zip'
VIDEO_FOLDER = WORKSPACE / 'data' / 'Coral_video'
VIDEO_WORK = CACHE_ROOT / 'CoralGuardX_AccuracyFirst_Video'
INPUT_ROOT = VIDEO_WORK / 'input'

if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)
INPUT_ROOT.mkdir(parents=True, exist_ok=True)

if VIDEO_ZIP.exists():
    with zipfile.ZipFile(VIDEO_ZIP, 'r') as z:
        z.extractall(INPUT_ROOT)
else:
    for p in VIDEO_FOLDER.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.mp4','.mov','.avi','.mkv','.m4v','.webm','.mpg','.mpeg'}:
            shutil.copy2(p, INPUT_ROOT / p.name)

VIDEO_EXTS={'.mp4','.mov','.avi','.mkv','.m4v','.webm','.mpg','.mpeg'}
video_paths=sorted([p for p in INPUT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS])
if VIDEO_LIMIT is not None:
    video_paths=video_paths[:int(VIDEO_LIMIT)]
if not video_paths:
    raise RuntimeError('No videos found')


def extract_frames(video_path, frame_dir):
    if frame_dir.exists():
        shutil.rmtree(frame_dir)
    frame_dir.mkdir(parents=True, exist_ok=True)
    cap=cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {video_path}')
    fps=float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps<=0:
        fps=25.0
    W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n=0
    while True:
        if VIDEO_MAX_FRAMES is not None and n>=int(VIDEO_MAX_FRAMES):
            break
        ok,frame=cap.read()
        if not ok:
            break
        cv2.imwrite(str(frame_dir/f'{n:06d}.jpg'), frame, [int(cv2.IMWRITE_JPEG_QUALITY),95])
        n+=1
    cap.release()
    return n,fps,W,H


def center(box):
    x1,y1,x2,y2=map(float,box)
    return np.array([(x1+x2)/2,(y1+y2)/2],dtype=np.float32)


def associate_tracks(tracks,dets,fi,W,H):
    active=[tid for tid,t in tracks.items() if fi-t['last_frame']<=TRACK_MAX_GAP]
    if not active:
        return [],list(range(len(dets)))
    if not dets:
        return [],[]

    diag=max(math.hypot(W,H),1.0)
    scores=np.zeros((len(active),len(dets)),dtype=np.float32)
    for i,tid in enumerate(active):
        t=tracks[tid]
        for j,d in enumerate(dets):
            mi=mask_iou(t['last_mask'],d['mask'])
            bi=box_iou(t['last_box'],d['box'])
            cd=float(np.linalg.norm(center(t['last_box'])-center(d['box']))/diag)
            cs=max(0.0,1.0-5.0*cd)
            scores[i,j]=0.50*mi+0.35*bi+0.15*cs

    rr,cc=linear_sum_assignment(1.0-scores)
    matches=[]; used=set()
    for r,c in zip(rr,cc):
        tid=active[r]; d=dets[c]
        mi=mask_iou(tracks[tid]['last_mask'],d['mask'])
        bi=box_iou(tracks[tid]['last_box'],d['box'])
        if mi<TRACK_MASK_IOU_MIN and bi<TRACK_BOX_IOU_MIN:
            continue
        matches.append((tid,c)); used.add(c)
    unmatched=[j for j in range(len(dets)) if j not in used]
    return matches,unmatched

print('Videos:')
for p in video_paths: print(' -',p.name)
print('STATUS: VIDEO_INPUT_READY')


In [ ]:
# CELL 14 — YOLO26m + CORALSCOP SEED PASS ON VIDEO REFRESH FRAMES

SEED_ROOT = VIDEO_WORK / 'seeds'
SEED_ROOT.mkdir(parents=True, exist_ok=True)
video_seed_metadata={}

for vi,vp in enumerate(video_paths,1):
    stem=vp.stem.replace(' ','_')
    frame_dir=VIDEO_WORK/stem/'frames'
    seed_dir=SEED_ROOT/stem
    if seed_dir.exists(): shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True,exist_ok=True)

    print('\n'+'='*110)
    print(f'SEED PASS {vi}/{len(video_paths)}: {vp.name}')
    print('='*110)

    n_frames,fps,W,H=extract_frames(vp,frame_dir)
    tracks={}; next_id=1; records=[]
    refresh=list(range(0,n_frames,REFRESH_STRIDE))
    spatial_count=0; spatial_time=0.0

    for ri,fi in enumerate(refresh,1):
        bgr=cv2.imread(str(frame_dir/f'{fi:06d}.jpg'))
        if bgr is None: continue
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

        t0=time.time()
        dets=final_spatial_instances(rgb)
        spatial_time += time.time()-t0

        # Do not seed obvious non-coral tracks into SAM2.
        dets=[d for d in dets if d['label']!='Non-Coral']
        spatial_count += len(dets)

        matches,unmatched=associate_tracks(tracks,dets,fi,W,H)
        for tid,di in matches:
            d=dets[di]; t=tracks[tid]
            t['last_frame']=fi; t['last_box']=d['box'].copy(); t['last_mask']=d['mask'].copy(); t['hits']+=1; t['score']+=d['det_conf']
            mp=seed_dir/f'track_{tid:03d}_frame_{fi:06d}.png'
            cv2.imwrite(str(mp),d['mask'].astype(np.uint8)*255)
            records.append({'track_id':tid,'frame_idx':fi,'mask_path':str(mp),'label':d['label'],'cls_conf':float(d['cls_conf'])})

        for di in unmatched:
            d=dets[di]; tid=next_id; next_id+=1
            tracks[tid]={'last_frame':fi,'last_box':d['box'].copy(),'last_mask':d['mask'].copy(),'hits':1,'score':d['det_conf']}
            mp=seed_dir/f'track_{tid:03d}_frame_{fi:06d}.png'
            cv2.imwrite(str(mp),d['mask'].astype(np.uint8)*255)
            records.append({'track_id':tid,'frame_idx':fi,'mask_path':str(mp),'label':d['label'],'cls_conf':float(d['cls_conf'])})

        if ri%20==0 or ri==len(refresh):
            print(f'Refresh {ri}/{len(refresh)} | frame={fi}/{n_frames} | tracks={len(tracks)} | masks={spatial_count}')

    ranked=sorted(tracks.items(), key=lambda kv:(kv[1]['hits'],kv[1]['score']/max(kv[1]['hits'],1)), reverse=True)
    selected={tid for tid,_ in ranked[:MAX_SAM2_TRACKS]}
    records=[r for r in records if r['track_id'] in selected]

    meta={
        'video':str(vp),'frame_dir':str(frame_dir),'n_frames':int(n_frames),'fps':float(fps),'width':int(W),'height':int(H),
        'refresh_stride':int(REFRESH_STRIDE),'candidate_conf':float(DET_CANDIDATE_CONF),'selected_tracks':int(len(selected)),
        'seed_records':records,'spatial_instances':int(spatial_count),
        'avg_spatial_seconds_per_refresh':float(spatial_time/max(len(refresh),1)),
    }
    (seed_dir/'seed_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
    video_seed_metadata[stem]=meta
    print('Selected tracks:',len(selected),' Seed records:',len(records))

print('STATUS: VIDEO_CORALSCOP_SEEDS_READY')


In [ ]:
# CELL 15 — LOAD SAM2.1 TEMPORAL MEMORY

try:
    coralscop_predictor.reset_image()
except Exception:
    pass

# Seed pass is complete, so release large spatial models from GPU.
coralscop_model.to('cpu')
try:
    detector.to('cpu')
except Exception:
    pass

gc.collect(); torch.cuda.empty_cache()

from sam2.sam2_video_predictor import SAM2VideoPredictor
sam2_predictor = SAM2VideoPredictor.from_pretrained(SAM2_MODEL_ID)

print('SAM2:', type(sam2_predictor).__name__)
print('STATUS: SAM2_READY')


In [ ]:
# CELL 16 — SAM2 PROPAGATION + DUAL-VIEW EMA + FINAL H.264 VIDEO

FFMPEG=shutil.which('ffmpeg')
if FFMPEG is None:
    raise RuntimeError('FFmpeg not found on PATH')

video_summaries=[]

for vi,vp in enumerate(video_paths,1):
    stem=vp.stem.replace(' ','_')
    meta=video_seed_metadata[stem]
    frame_dir=Path(meta['frame_dir'])
    n_frames=int(meta['n_frames']); fps=float(meta['fps']); W=int(meta['width']); H=int(meta['height'])
    records=meta['seed_records']
    if not records:
        print('SKIP no seeds:',vp.name); continue

    out_dir=VIDEO_ROOT/stem; out_dir.mkdir(parents=True,exist_ok=True)
    final_video=out_dir/f'{stem}_ACCURACY_FIRST_FINAL.mp4'
    final_json=out_dir/f'{stem}_ACCURACY_FIRST_summary.json'

    print('\n'+'='*110)
    print(f'FINAL PASS {vi}/{len(video_paths)}: {vp.name}')
    print('='*110)

    state=sam2_predictor.init_state(
        video_path=str(frame_dir),
        offload_video_to_cpu=True,
        offload_state_to_cpu=False,
        async_loading_frames=False,
    )

    # Repeated CoralSCOP masks become SAM2 conditioning corrections.
    for rec in records:
        m=cv2.imread(rec['mask_path'],cv2.IMREAD_GRAYSCALE)
        if m is None: continue
        m=normalize_mask_2d(m)
        sam2_predictor.add_new_mask(
            inference_state=state,
            frame_idx=int(rec['frame_idx']),
            obj_id=int(rec['track_id']),
            mask=m,
        )

    cmd=[
        FFMPEG,'-loglevel','error','-y',
        '-f','rawvideo','-vcodec','rawvideo','-pix_fmt','bgr24','-s',f'{W}x{H}','-r',f'{fps:.8f}','-i','pipe:0',
        '-i',str(vp),'-map','0:v:0','-map','1:a:0?',
        '-c:v','libx264','-preset','slow','-crf','12','-pix_fmt','yuv420p',
        '-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(final_video)
    ]
    ff=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)

    cls_state={}; transitions=0; switches=0; propagated=0; frames_done=0; t0=time.time()

    with torch.inference_mode(), torch.autocast('cuda',dtype=torch.bfloat16):
        for out_fi,out_ids,out_logits in sam2_predictor.propagate_in_video(state):
            fi=int(out_fi)
            if fi>=n_frames: continue
            bgr=cv2.imread(str(frame_dir/f'{fi:06d}.jpg'))
            if bgr is None: continue
            rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

            instances=[]; ids=[]
            for oi,obj_id in enumerate(out_ids):
                lg=out_logits[oi]
                while lg.ndim>2: lg=lg[0]
                mask=(lg>0.0).detach().cpu().numpy().astype(bool)
                if int(mask.sum())<CORALSCOP_MIN_PIXELS: continue
                box=safe_mask_bbox(mask)
                if box is None: continue
                instances.append({'mask':mask,'box':box}); ids.append(int(obj_id))
            propagated += len(instances)

            if instances and fi%CLS_STRIDE==0:
                classified=classify_instances(rgb,instances)
                if len(classified)==len(instances):
                    for obj_id,inst in zip(ids,classified):
                        p=inst['prob_vector']; s=cls_state.get(obj_id)
                        if s is None:
                            s={'prob':p.copy(),'last_label':None}
                        else:
                            smooth=SPECIES_EMA_ALPHA*p+(1-SPECIES_EMA_ALPHA)*s['prob']
                            smooth/=max(float(smooth.sum()),1e-9); s['prob']=smooth

                        order=np.argsort(s['prob'])[::-1]; top,second=int(order[0]),int(order[1])
                        raw=IDX_TO_CLASS[top]; conf=float(s['prob'][top]); margin=float(s['prob'][top]-s['prob'][second])
                        non_prob=float(s['prob'][CLASS_TO_IDX['NON_CORAL']])
                        if raw=='NON_CORAL' or non_prob>=float(cal['non_coral_threshold']):
                            label='Non-Coral'
                        elif conf<float(cal['known_conf_threshold']) or margin<float(cal['known_margin_threshold']) or inst['label']=='Other/Unknown Coral':
                            label='Other/Unknown Coral'
                        else:
                            label=raw

                        if s['last_label'] is not None:
                            transitions+=1; switches += int(label!=s['last_label'])
                        s['last_label']=label; s['label']=label; s['confidence']=conf; cls_state[obj_id]=s

            vis=bgr.copy(); overlay=bgr.copy()
            for obj_id,inst in zip(ids,instances):
                s=cls_state.get(obj_id,{'label':'Analyzing Coral','confidence':0.0})
                label=s['label']; conf=float(s.get('confidence',0.0))
                color=(150,150,150) if label=='Non-Coral' else ((0,165,255) if label=='Other/Unknown Coral' else (0,255,0))
                mask=inst['mask']
                overlay[mask]=(0.55*overlay[mask]+0.45*np.asarray(color,dtype=np.float32)).astype(np.uint8)
                contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(vis,contours,-1,color,2)
                x1,y1,x2,y2=[int(round(z)) for z in inst['box']]
                cv2.putText(vis,f'ID {obj_id} | {label} | {conf:.2f}',(x1,max(25,y1-7)),cv2.FONT_HERSHEY_SIMPLEX,0.65,color,2,cv2.LINE_AA)

            vis=cv2.addWeighted(overlay,0.42,vis,0.58,0)
            ff.stdin.write(vis.tobytes()); frames_done+=1
            if frames_done%100==0:
                print(f'Rendered {frames_done}/{n_frames} | masks={propagated}')

    ff.stdin.close(); ff.wait()
    err=ff.stderr.read().decode('utf-8',errors='ignore')
    if ff.returncode!=0:
        raise RuntimeError(err[-5000:])

    summary={
        'video':vp.name,'frames':n_frames,'detector':str(DETECTOR_TEACHER),'candidate_conf':DET_CANDIDATE_CONF,
        'coralscop':str(CORALSCOP_CKPT),'refresh_stride':REFRESH_STRIDE,'sam2':SAM2_MODEL_ID,
        'selected_tracks':meta['selected_tracks'],'seed_records':len(records),'propagated_masks':propagated,
        'dual_view_classifier':str(BEST_DUAL),'label_transitions':transitions,'label_switches':switches,
        'species_flicker_rate':float(switches/transitions) if transitions else 0.0,
        'final_pass_seconds':float(time.time()-t0),'output':str(final_video),
    }
    final_json.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
    video_summaries.append(summary)
    print('Saved:',final_video)

    sam2_predictor.reset_state(state); del state; gc.collect(); torch.cuda.empty_cache()

print('STATUS: VIDEO_TEST_COMPLETE')


In [ ]:
# CELL 17 — FINAL SUMMARY

summary_df=pd.DataFrame(video_summaries)
SUMMARY_CSV=VIDEO_ROOT/'accuracy_first_video_summary.csv'
summary_df.to_csv(SUMMARY_CSV,index=False)

FINAL_JSON=FINAL_ROOT/'ACCURACY_FIRST_FINAL_PROJECT_SUMMARY.json'
final_summary={
    'architecture':'YOLO26m old detector -> CoralSCOP every candidate on refresh frames -> duplicate resolver -> Dual-View EfficientNetV2-S -> OOD/Non-Coral -> SAM2.1 temporal memory',
    'detector':str(DETECTOR_TEACHER),'candidate_conf':DET_CANDIDATE_CONF,'coralscop':str(CORALSCOP_CKPT),
    'dual_view_classifier':str(BEST_DUAL),'dual_view_validation':{k:float(v) for k,v in val_metrics.items()},
    'sam2':SAM2_MODEL_ID,'refresh_stride':REFRESH_STRIDE,'videos_processed':len(video_summaries),
    'held_out_detector_test_used':False,
    'limitations':[
        'Other/Unknown Coral is pragmatic rejection; no explicit unseen-species validation set.',
        'Dead Coral and Algae are not explicit current training classes.',
        'Video has no frame-level GT, so no invented video accuracy is reported.'
    ]
}
FINAL_JSON.write_text(json.dumps(final_summary,indent=2,ensure_ascii=False),encoding='utf-8')

print('='*110)
print('ACCURACY-FIRST COMPLETE')
print('='*110)
print('Videos:',VIDEO_ROOT)
print('CSV:',SUMMARY_CSV)
print('JSON:',FINAL_JSON)
print('Held-out detector TEST: NOT USED')
print('STATUS: COMPLETE')


## تشغيل النوتبوك

شغّل الخلايا بالترتيب من **Cell 1 إلى Cell 17**.

أول تجربة اترك:

```python
REFRESH_STRIDE = 3
```

ولو أردت بعد نجاح التجربة أقصى refresh ممكن:

```python
REFRESH_STRIDE = 1
```

الـoutput النهائي سيكون تحت:

```text
D:\Coral_Local_Workspace\outputs\CoralGuard_X_ACCURACY_FIRST\06_Final_Videos
```
